# Assignment 7 - Baseline models

Short, reproducible notebook for baseline models. The logic remains aligned with Assignments 1-6: target `hospital_use_per_1000`, F0/F1 feature sets, seed 42, PracticeCode-aware split, R2/RMSE/MAE, and no preprocessing fitted outside training folds.

In [ ]:
from pathlib import Path
import hashlib, json, math, platform, sys
import numpy as np
import pandas as pd

SEED, TEST_SIZE, CV_SPLITS = 42, 0.2, 5
TARGET, GROUP = "hospital_use_per_1000", "PracticeCode"
ROOT = Path.cwd()
if ROOT.name == "assignment7":
    ROOT = ROOT.parent
OUT = Path(globals().get("OUTPUT_DIR", ROOT / "assignment7" / "outputs"))
if not OUT.is_absolute():
    OUT = ROOT / OUT
OUT.mkdir(parents=True, exist_ok=True)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def clean_json(value):
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()}
    if isinstance(value, list):
        return [clean_json(v) for v in value]
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    return value

In [ ]:
raw_dir = ROOT / "assignment4" / "data" / "raw"
dem_path = raw_dir / "sample_gp_practice_population_demographics.csv"
sup_path = raw_dir / "sample_gp_practice_supporting_inputs.csv"
dem = pd.read_csv(dem_path, parse_dates=["Date"])
sup = pd.read_csv(sup_path, parse_dates=["Date"])
dem[GROUP] = dem[GROUP].astype(str)
sup[GROUP] = sup[GROUP].astype(str)

age_cols = ["AllAges", "Ages0to4", "Ages5to14", "Ages15to24", "Ages25to44", "Ages45to64", "Ages65to74", "Ages75to84", "Ages85plus"]
dem[age_cols] = dem[age_cols].apply(pd.to_numeric, errors="coerce")
keys = ["Date", GROUP]
valid_keys = dem.groupby(keys)["Sex"].agg(lambda s: {"All", "Female"}.issubset(set(s)))
valid_keys = valid_keys[valid_keys].index

base = dem.set_index(keys).loc[valid_keys].reset_index()
all_rows = base[base["Sex"].eq("All")].copy()
female = base[base["Sex"].eq("Female")][keys + ["AllAges"]].rename(columns={"AllAges": "female_population"})
df = all_rows[keys + ["HB", "HSCP"] + age_cols].merge(female, on=keys, validate="1:1")
df = df[df["AllAges"].fillna(0).gt(0)].copy()
df["share_age_0_14"] = (df["Ages0to4"] + df["Ages5to14"]) / df["AllAges"]
df["share_age_65_plus"] = (df["Ages65to74"] + df["Ages75to84"] + df["Ages85plus"]) / df["AllAges"]
df["share_female"] = df["female_population"] / df["AllAges"]
df["log_all_ages"] = np.log1p(df["AllAges"])
df = df.merge(sup[keys + ["gp_availability", "deprivation_index", TARGET]], on=keys, validate="1:1")
df = df.sort_values(["Date", GROUP]).reset_index(drop=True)
df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")
df.to_csv(OUT / "analytical_dataset_assignment7.csv", index=False)
df.head()

In [ ]:
rng = np.random.default_rng(SEED)
groups = np.array(sorted(df[GROUP].unique()))
test_n = max(1, math.ceil(TEST_SIZE * len(groups)))
test_groups = set(rng.permutation(groups)[:test_n])
is_test = df[GROUP].isin(test_groups)
train, test = df[~is_test].reset_index(drop=True), df[is_test].reset_index(drop=True)

fold_count = min(CV_SPLITS, train[GROUP].nunique())
fold_groups = np.array_split(np.random.default_rng(SEED).permutation(sorted(train[GROUP].unique())), fold_count)
folds = [(train[~train[GROUP].isin(g)].copy(), train[train[GROUP].isin(g)].copy()) for g in fold_groups]
df[["Date", GROUP]].assign(split=np.where(is_test, "test", "train")).to_csv(OUT / "split_assignments.csv", index=False)
sorted(test_groups), fold_count

In [ ]:
def design(fit_df, apply_df, numeric, categorical):
    med = fit_df[numeric].median()
    x_fit = fit_df[numeric].fillna(med).astype(float)
    mean, std = x_fit.mean(), x_fit.std(ddof=0).replace(0, 1)
    parts = [((apply_df[numeric].fillna(med).astype(float) - mean) / std).reset_index(drop=True)]
    for col in categorical:
        mode = fit_df[col].mode(dropna=True)
        mode = mode.iloc[0] if len(mode) else "__missing__"
        levels = sorted(fit_df[col].fillna(mode).astype(str).unique())
        cat = pd.Series(pd.Categorical(apply_df[col].fillna(mode).astype(str), categories=levels), name=col)
        parts.append(pd.get_dummies(cat, prefix=col).astype(float).reset_index(drop=True))
    x = pd.concat(parts, axis=1)
    return x.to_numpy(float), list(x.columns)

def metrics(y, pred):
    err = y - pred
    denom = float(((y - y.mean()) ** 2).sum())
    return {"r2": round(1 - float((err ** 2).sum()) / denom, 6) if denom else None,
            "rmse": round(math.sqrt(float((err ** 2).mean())), 6),
            "mae": round(float(np.abs(err).mean()), 6)}

def fit_linear(x, y, alpha=0.0):
    z = np.c_[np.ones(len(x)), x]
    penalty = np.eye(z.shape[1]) * alpha
    penalty[0, 0] = 0
    return np.linalg.pinv(z.T @ z + penalty) @ z.T @ y

def predict_linear(beta, x):
    return np.c_[np.ones(len(x)), x] @ beta

def fit_tree(x, y, max_depth=3, min_leaf=2):
    def build(x, y, depth):
        pred = float(y.mean())
        if depth >= max_depth or len(y) < 2 * min_leaf or np.allclose(y, y[0]):
            return (pred, None, None, None, None)
        best = None
        for j in range(x.shape[1]):
            vals = np.unique(x[:, j])
            for t in (vals[:-1] + vals[1:]) / 2:
                left = x[:, j] <= t
                if left.sum() >= min_leaf and (~left).sum() >= min_leaf:
                    loss = ((y[left] - y[left].mean()) ** 2).sum() + ((y[~left] - y[~left].mean()) ** 2).sum()
                    if best is None or loss < best[0]:
                        best = (loss, j, t, left)
        if best is None:
            return (pred, None, None, None, None)
        _, j, t, left = best
        return (pred, j, t, build(x[left], y[left], depth + 1), build(x[~left], y[~left], depth + 1))
    return build(x, y, 0)

def predict_tree(tree, x):
    def one(row, node):
        pred, j, t, left, right = node
        return pred if j is None else one(row, left if row[j] <= t else right)
    return np.array([one(row, tree) for row in x])

In [ ]:
F0 = ["log_all_ages", "gp_availability"]
F1 = F0 + ["share_age_65_plus", "share_female", "deprivation_index"]
models = [
    dict(name="dummy_mean_f0", display_name="Dummy mean baseline", family="dummy", feature_set="F0", num=F0, cat=[], params=[{}], strategy="No tuning; predicts the training-set mean."),
    dict(name="ols_f0", display_name="OLS baseline", family="linear", feature_set="F0", num=F0, cat=[], params=[{"alpha": 0.0}], strategy="No tuning; ordinary least squares."),
    dict(name="ridge_f1", display_name="Ridge extended baseline", family="linear", feature_set="F1", num=F1, cat=["HB", "HSCP"], params=[{"alpha": a} for a in [0.1, 1.0, 10.0]], strategy="Limited alpha grid selected by group-aware CV on training folds only."),
    dict(name="decision_tree_f1", display_name="Decision Tree benchmark", family="tree", feature_set="F1", num=F1, cat=["HB", "HSCP"], params=[{"max_depth": 3, "min_leaf": 2}], strategy="No tuning; conservative tree depth and leaf size fixed before test evaluation."),
]

def evaluate(spec, tr, te):
    candidates = []
    for params in spec["params"]:
        fold_scores = []
        for ftr, fva in folds:
            xtr, _ = design(ftr, ftr, spec["num"], spec["cat"])
            xva, _ = design(ftr, fva, spec["num"], spec["cat"])
            ytr, yva = ftr[TARGET].to_numpy(float), fva[TARGET].to_numpy(float)
            if spec["family"] == "dummy":
                pred = np.repeat(ytr.mean(), len(yva))
            elif spec["family"] == "linear":
                pred = predict_linear(fit_linear(xtr, ytr, params["alpha"]), xva)
            else:
                pred = predict_tree(fit_tree(xtr, ytr, **params), xva)
            fold_scores.append(metrics(yva, pred))
        cv = {"folds": len(fold_scores)}
        for m in ["r2", "rmse", "mae"]:
            vals = np.array([s[m] for s in fold_scores if s[m] is not None], float)
            cv[f"mean_{m}"] = round(float(vals.mean()), 6) if len(vals) else None
            cv[f"std_{m}"] = round(float(vals.std(ddof=1)), 6) if len(vals) > 1 else 0.0
        candidates.append({"params": params, "cv_summary": cv, "fold_metrics": fold_scores})
    selected = min(candidates, key=lambda c: c["cv_summary"]["mean_rmse"])

    xtr, cols = design(tr, tr, spec["num"], spec["cat"])
    xte, _ = design(tr, te, spec["num"], spec["cat"])
    ytr, yte = tr[TARGET].to_numpy(float), te[TARGET].to_numpy(float)
    if spec["family"] == "dummy":
        pred = np.repeat(ytr.mean(), len(yte))
    elif spec["family"] == "linear":
        pred = predict_linear(fit_linear(xtr, ytr, selected["params"]["alpha"]), xte)
    else:
        pred = predict_tree(fit_tree(xtr, ytr, **selected["params"]), xte)
    pred_rows = te[["Date", GROUP, TARGET]].copy()
    pred_rows["model_name"] = spec["name"]
    pred_rows["prediction"] = pred.round(6)
    pred_rows["residual"] = (pred_rows[TARGET] - pred_rows["prediction"]).round(6)
    return {"model_name": spec["name"], "display_name": spec["display_name"], "family": spec["family"],
            "feature_set": spec["feature_set"], "selected_params": selected["params"], "tuning_strategy": spec["strategy"],
            "encoded_feature_count": len(cols), "holdout": metrics(yte, pred), "cross_validation": selected["cv_summary"],
            "candidate_results": candidates}, pred_rows

In [ ]:
results, predictions = zip(*(evaluate(spec, train, test) for spec in models))
rows = []
for r in results:
    rows.append({"model_name": r["model_name"], "display_name": r["display_name"], "family": r["family"],
                 "feature_set": r["feature_set"], "selected_params": json.dumps(r["selected_params"], sort_keys=True),
                 "tuning_strategy": r["tuning_strategy"], "encoded_feature_count": r["encoded_feature_count"],
                 **{f"holdout_{k}": v for k, v in r["holdout"].items()},
                 **{f"cv_{k}": v for k, v in r["cross_validation"].items()}})
metrics_table = pd.DataFrame(rows)
ols = metrics_table[metrics_table.model_name.eq("ols_f0")].iloc[0]
comparison = metrics_table.copy()
comparison["delta_r2_vs_ols_f0"] = comparison["holdout_r2"] - ols["holdout_r2"]
comparison["rmse_reduction_pct_vs_ols_f0"] = ((ols["holdout_rmse"] - comparison["holdout_rmse"]) / ols["holdout_rmse"] * 100).round(6)
comparison = comparison.sort_values(["holdout_rmse", "model_name"]).reset_index(drop=True)

metrics_table.to_csv(OUT / "metrics_table.csv", index=False)
comparison.to_csv(OUT / "model_comparison.csv", index=False)
pd.concat(predictions, ignore_index=True).to_csv(OUT / "holdout_predictions.csv", index=False)
(OUT / "metrics_table.json").write_text(metrics_table.to_json(orient="records", indent=2), encoding="utf-8")
(OUT / "model_comparison.json").write_text(comparison.to_json(orient="records", indent=2), encoding="utf-8")
metadata = {
    "assignment": 7, "random_seed": SEED, "target_column": TARGET, "primary_metric": "holdout_rmse",
    "dataset": {"rows": len(df), "practices": int(df[GROUP].nunique()), "demographics_sha256": sha256(dem_path), "supporting_sha256": sha256(sup_path)},
    "holdout_split": {"split_type": "PracticeCode-aware holdout", "train_groups": sorted(set(groups) - test_groups), "test_groups": sorted(test_groups), "train_rows": len(train), "test_rows": len(test)},
    "cross_validation": {"planned_splits": CV_SPLITS, "actual_splits": fold_count, "split_type": "PracticeCode-aware CV on training partition"},
    "leakage_controls": ["PracticeCode groups are not split across train/test.", "CV is fitted only on the training partition.", "Imputation, scaling and one-hot levels are fitted separately in each training fold.", "Ridge alpha is selected without looking at the test set."],
    "environment": {"python": sys.version, "platform": platform.platform(), "numpy": np.__version__, "pandas": pd.__version__},
}
(OUT / "run_metadata.json").write_text(json.dumps(clean_json(metadata), indent=2), encoding="utf-8")
(OUT / "full_model_results.json").write_text(json.dumps(clean_json(list(results)), indent=2), encoding="utf-8")
RESULT = {"output_dir": str(OUT), "metrics_csv": str(OUT / "metrics_table.csv"), "comparison_csv": str(OUT / "model_comparison.csv"), "metadata_json": str(OUT / "run_metadata.json")}
comparison